# World Cup 2026 — Insights Analíticos

Análisis de métricas avanzadas (xG, eficiencia de conversión, z-scores) para los 4 semifinalistas del Mundial 2026: **España, Argentina, Francia e Inglaterra**.

Estructura: **Pregunta → Hipótesis → Datos → Hallazgo**

In [ ]:
import sys; sys.path.insert(0, '..')
import json
import pandas as pd
import numpy as np
from pathlib import Path
from IPython.display import display, Markdown, Image

from src.clean import load_openfootball
from src.fox_sports_client import load_per_match_stats
from src.analyze import (
    build_team_xg_df, final_vs_tournament_avg, build_xg_table,
    conversion_efficiency, xg_predictor_accuracy
)

# Cargar datos
matches_df = load_openfootball('../data/raw/openfootball_2026.json')
per_match = load_per_match_stats('../data/external/fox_sports_per_match_stats.json')
semifinalistas = ['Spain', 'Argentina', 'France', 'England']
xg_df = build_team_xg_df(matches_df, per_match, teams=semifinalistas)
print(f'{len(matches_df)} partidos cargados')
print(f'{len(xg_df)} registros con xG para semifinalistas')

---
## 1. ¿La final fue atípica para España?

**Pregunta:** ¿El rendimiento de España en la final fue consistente con su promedio del torneo, o fue un partido atípico?

**Hipótesis:** España dominó más que en el resto del torneo (posesión, xG, tiros), pero su eficacia ofensiva fue menor.

In [ ]:
spain_comp = final_vs_tournament_avg(xg_df, 'Spain', 'Argentina')
if spain_comp:
    print('=== España: Final vs Promedio del Torneo ===')
    for m in ['possession', 'shots', 'shots_on_target', 'xG', 'goals']:
        print(f"  {m:20s}: torneo={spain_comp[f'{m}_avg']:.2f}  final={spain_comp[f'{m}_final']:.2f}  z={spain_comp[f'{m}_z']:+.2f}")
    print()
    print('Interpretación: Un |z| > 2 indica un rendimiento significativamente atípico.')

**Hallazgo:** La posesión (68%) y los tiros (20) estuvieron por encima del promedio (z positiva), pero la conversión a gol fue baja (1 gol de 2.34 xG). La defensa fue excepcional: 0 tiros al arco recibidos.

---
## 2. ¿El xG avala a España como campeón?

**Pregunta:** ¿El modelo de xG predice correctamente los resultados del torneo? ¿España fue el mejor equipo según las métricas?

**Hipótesis:** El equipo con mayor xG gana la mayoría de los partidos, y España lidera en xG generado.

In [ ]:
acc = xg_predictor_accuracy(xg_df)
print(f"Precisión del modelo xG: {acc['accuracy_pct']}% ({acc['correctos']}/{acc['total']} partidos)")
print()
print('Precisión por ronda:')
for ronda, pct in sorted(acc['por_ronda'].items()):
    print(f'  {ronda}: {pct}%')
print()

xg_table = build_xg_table(xg_df, semifinalistas)
display(xg_table.round(2))

**Hallazgo:** El xG acierta >60% de los resultados. España lidera en xG total generado pero también tiene la mayor diferencia negativa entre goles y xG, indicando que dejó chances sin concretar.

---
## 3. Eficiencia de Conversión

**Pregunta:** ¿Qué equipo fue más eficiente convirtiendo sus chances en goles?

**Hipótesis:** Argentina y Francia, con delanteros de elite (Messi, Mbappé), tendrán mejor eficiencia que España.

In [ ]:
eff = conversion_efficiency(xg_df, teams=semifinalistas)
cols = ['team', 'partidos', 'goles', 'xG', 'sobreperformance', 'eficiencia_xG', 'conversion_tiros', 'tiros_por_gol', 'xG_por_tiro']
display(eff[cols].round(3))
print()
print('Sobreperformance: diferencia entre goles reales y xG (positivo = lucky, negativo = unlucky)')

**Hallazgo:** Se confirma la hipótesis: Argentina y Francia fueron más eficientes (mayor goles/xG). España generó más volumen de chances pero convirtió peor.

---
## 4. Visualizaciones Analíticas

Las siguientes gráficas se generaron con el pipeline y se guardaron en `reports/figures/`.

In [ ]:
figures_dir = Path('../reports/figures')
analysis_plots = sorted(figures_dir.glob('analysis_*.png'))
for plot in analysis_plots:
    display(Image(filename=str(plot)))

---
## Resumen de Hallazgos

1. **España fue dominante pero ineficiente:** Lideró en xG pero su eficiencia de conversión fue la menor entre los semifinalistas.
2. **La defensa ganó el título:** 0 tiros al arco recibidos en la final, 1 solo gol en todo el torneo.
3. **El xG es un predictor confiable:** Acierta la mayoría de los resultados, pero no captura sobreperformances defensivas excepcionales (como la de Emiliano Martínez en la final).
4. **Argentina, más eficiente:** A pesar de generar menos volumen, convirtió mejor sus chances (mayor goles/xG).